# DCASE 2026 Task 1 — World #1 Notebook

Single self-contained notebook for **Heterogeneous Audio Classification** on BSD10k.

**Goal:** maximize Hierarchical Accuracy (H-Acc).

**Cumulative experiments:**
| ID | Description | Expected effect |
|----|-------------|-----------------|
| EXP-000 | baseline (reference) | H-Acc 79.45% |
| EXP-001 | + confidence ≥ 4 filter | +9% 2nd / +5% top (paper A) |
| EXP-005 | + L_Top + L_Contr losses | structure latent space (paper B) |
| EXP-006 | + hidden_size 256 | more capacity |
| EXP-007 | + hidden_size 512 | largest capacity |

**Architecture:** audio_emb + text_emb → EmbeddingEncoder × 2 → AttentionFusion → latent_projector → residual_classifier → 23 classes.  
**Loss (when hier_loss=True):** `L_total = L_CE(2nd) + λ_top·L_Top + λ_contr·L_Contr`.  
**References:** Anastasopoulou et al. DCASE 2025; Khosla et al. SupCon 2020.

## 1. Setup, paths, seed

In [ ]:
import os
import sys
import json
import random
import collections.abc
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit

PROJECT_ROOT = r'C:/Users/solok/Desktop/Dcase baseline'
CODE_DIR = os.path.join(PROJECT_ROOT, 'dcase2026_task1_baseline')
DATA_BUILD_DIR = os.path.join(CODE_DIR, 'data')
EXPERIMENTS_DIR = os.path.join(PROJECT_ROOT, 'experiments')
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', sys.version.split()[0])
print('Torch :', torch.__version__)
print('Device:', DEVICE, '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

def set_seed(seed=1821):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return seed

SEED = set_seed(1821)
print('seed =', SEED)

DATASET_NAME = 'BSD10k-v1.2'
METADATA_CSV = os.path.join(PROJECT_ROOT, 'data/metadata/BSD10k_metadata.csv')
PROCESSED_CSV = os.path.join(DATA_BUILD_DIR, 'processed_dataset.csv')
with open(os.path.join(DATA_BUILD_DIR, 'class_dict.json'), 'r') as f:
    CLASS_DICT = json.load(f)
with open(os.path.join(DATA_BUILD_DIR, 'top_class_dict.json'), 'r') as f:
    TOP_CLASS_DICT = json.load(f)
NUM_CLASSES = len(CLASS_DICT)
NUM_TOP_CLASSES = len(TOP_CLASS_DICT)
print(f'classes: {NUM_CLASSES} (2nd-level), {NUM_TOP_CLASSES} (top-level)')

## 2. Helper utilities

In [ ]:
def build_class_to_topclass_mapping(class_dict, top_class_dict):
    m = {}
    for c, cid in class_dict.items():
        for t, tid in top_class_dict.items():
            if c.startswith(t):
                m[cid] = tid; break
    return m

def build_class_to_topclass_tensor(class_dict, top_class_dict, device):
    n = len(class_dict)
    out = torch.zeros(n, dtype=torch.long, device=device)
    for c, cid in class_dict.items():
        for t, tid in top_class_dict.items():
            if c.startswith(t):
                out[cid] = tid; break
    return out

def build_id_to_class_mapping(class_dict):
    return {cid: c for c, cid in class_dict.items()}

def extend_subcat(s):
    if '-' not in s:
        raise Exception('invalid subcat: ' + s)
    return (s.split('-')[0], s)

def get_top_level(s):
    return extend_subcat(s)[0]

def intersection(a, b):
    return list(set(a).intersection(b))

def make_serializable(obj, decimals=6):
    if isinstance(obj, torch.Tensor):
        return make_serializable(obj.detach().cpu().numpy(), decimals)
    if isinstance(obj, np.ndarray):
        return round(float(obj), decimals) if obj.ndim == 0 else [make_serializable(x, decimals) for x in obj]
    if isinstance(obj, float):
        return round(obj, decimals)
    if isinstance(obj, int):
        return obj
    if isinstance(obj, collections.abc.Mapping):
        return {k: make_serializable(v, decimals) for k, v in obj.items()}
    if isinstance(obj, collections.abc.Iterable) and not isinstance(obj, (str, bytes)):
        return [make_serializable(x, decimals) for x in obj]
    return obj

CLASS_TO_TOPCLASS = build_class_to_topclass_mapping(CLASS_DICT, TOP_CLASS_DICT)
SUB2TOP_TENSOR = build_class_to_topclass_tensor(CLASS_DICT, TOP_CLASS_DICT, DEVICE)

## 3. Dataset — HATRDataset
Loads precomputed CLAP audio + text 512-d embeddings (.npy) per sound_id. Augmentation: Gaussian noise + random feature masking on training only.

In [ ]:
class HATRDataset(Dataset):
    def __init__(self, dataframe, aug=True, mask_pct=0.7):
        self.df = dataframe; self.aug = aug; self.mask_pct = mask_pct

    def _rand_mask(self, emb):
        max_m = int(emb.shape[0] * self.mask_pct)
        n = random.randint(1, max_m)
        idx = torch.randperm(emb.shape[0])[:n]
        m = torch.ones_like(emb); m[idx] = 0.0
        return emb * m

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        s = self.df.iloc[i]
        a = torch.tensor(np.load(s['audio_emb_filepath']), dtype=torch.float32)
        t = torch.tensor(np.load(s['text_emb_filepath']), dtype=torch.float32)
        if self.aug:
            a = a + torch.randn_like(a) * 1e-4; a = self._rand_mask(a)
            t = t + torch.randn_like(t) * 1e-4; t = self._rand_mask(t)
        return {
            'sound_id': s['index'],
            'audio_embedding': a,
            'text_embedding': t,
            'class': s['class'],
            'class_idx': int(s['class_idx']),
            'top_class': s['top_class'],
            'top_class_idx': int(s['top_class_idx']),
        }

## 4. Model — BaseClassifier (HATR architecture)
audio + text → per-modality EmbeddingEncoder → AttentionFusion → latent_projector → 2× ResidualBlock → class_predictor.  
Returns `(z, class_logit, attn_scores)` — `z` is reused by L_Contr.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, input_size, hidden_size, dropout=0.2, use_batch_norm=True):
        super().__init__()
        self.use_bn = use_batch_norm
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, input_size)
        self.act = nn.LeakyReLU(); self.drop = nn.Dropout(dropout)
        if use_batch_norm:
            self.norm1 = nn.BatchNorm1d(hidden_size)
            self.norm2 = nn.BatchNorm1d(input_size)

    def forward(self, x):
        r = x
        out = self.linear1(x)
        if self.use_bn: out = self.norm1(out)
        out = self.act(out); out = self.drop(out)
        out = self.linear2(out)
        if self.use_bn: out = self.norm2(out)
        out = out + r
        return self.act(out)

class EmbeddingEncoder(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.2, use_batch_norm=True, num_residual_blocks=3):
        super().__init__()
        h = max(input_size, output_size * 2)
        self.input_proj = nn.Sequential(nn.Linear(input_size, h), nn.LeakyReLU(), nn.Dropout(dropout))
        self.blocks = nn.ModuleList([ResidualBlock(h, h*2, dropout, use_batch_norm) for _ in range(num_residual_blocks)])
        self.output_proj = nn.Sequential(
            nn.Linear(h, h//2), nn.LeakyReLU(), nn.Dropout(dropout),
            nn.Linear(h//2, output_size),
        )
        self.use_bn = use_batch_norm
        if use_batch_norm:
            self.input_norm = nn.BatchNorm1d(input_size)
            self.output_norm = nn.BatchNorm1d(output_size)

    def forward(self, x):
        if self.use_bn: x = self.input_norm(x)
        x = self.input_proj(x)
        for b in self.blocks: x = b(x)
        x = self.output_proj(x)
        if self.use_bn: x = self.output_norm(x)
        return x

class AttentionFusion(nn.Module):
    def __init__(self, feature_size, dropout=0.2):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(feature_size*2, feature_size), nn.Tanh(),
            nn.Linear(feature_size, 2), nn.Softmax(dim=-1),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, a, t):
        c = torch.cat([a, t], dim=-1)
        w = self.attn(c)
        return self.drop(a * w[:, 0:1] + t * w[:, 1:2]), w

class BaseClassifier(nn.Module):
    def __init__(self, hidden_size=128, num_classes=23, emb_size_audio=0, emb_size_text=0,
                 dropout=0.1, use_batch_norm=True, mode='both',
                 num_residual_blocks=3, use_attention_fusion=True):
        super().__init__()
        self.hidden_size = hidden_size; self.num_classes = num_classes
        self.emb_size_audio = emb_size_audio; self.emb_size_text = emb_size_text
        self.dropout = dropout; self.use_batch_norm = use_batch_norm; self.mode = mode
        self.use_attention_fusion = use_attention_fusion and mode == 'both'

        self.audio_emb_extractor = EmbeddingEncoder(emb_size_audio, hidden_size, dropout, use_batch_norm, num_residual_blocks) if mode in ['audio', 'both'] else None
        self.text_emb_extractor = EmbeddingEncoder(emb_size_text, hidden_size, dropout, use_batch_norm, num_residual_blocks) if mode in ['text', 'both'] else None

        if mode == 'both':
            combined = hidden_size if self.use_attention_fusion else hidden_size*2
            self.fusion = AttentionFusion(hidden_size, dropout) if self.use_attention_fusion else None
        else:
            combined = hidden_size

        self.latent_projector = nn.Sequential(
            nn.Linear(combined, hidden_size*2), nn.LeakyReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_size*2, hidden_size), nn.LeakyReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size//2), nn.LeakyReLU(), nn.Dropout(dropout/2),
        )
        self.residual_classifier = nn.ModuleList([ResidualBlock(hidden_size//2, hidden_size, dropout/2, use_batch_norm) for _ in range(2)])
        self.class_predictor = nn.Sequential(
            nn.Linear(hidden_size//2, hidden_size//4), nn.LeakyReLU(), nn.Dropout(dropout/4),
            nn.Linear(hidden_size//4, num_classes),
        )

    def forward(self, audio_emb=None, text_emb=None):
        feats = []
        if self.audio_emb_extractor is not None: feats.append(self.audio_emb_extractor(audio_emb))
        if self.text_emb_extractor is not None: feats.append(self.text_emb_extractor(text_emb))
        if len(feats) > 1 and self.use_attention_fusion:
            x, attn = self.fusion(feats[0], feats[1])
        elif len(feats) > 1:
            x = torch.cat(feats, dim=-1); attn = None
        else:
            x = feats[0]; attn = None
        z = self.latent_projector(x)
        for b in self.residual_classifier: z = b(z)
        return z, self.class_predictor(z), attn

def init_weights(m):
    if isinstance(m, nn.Conv2d): nn.init.kaiming_normal_(m.weight, mode='fan_out')
    elif isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight)

## 5. Losses
**Plain CE** (EXP-000, 001) and **HierarchicalLoss** (EXP-005+).  
L_Top: differentiable surrogate — aggregate fine-class probs to top via membership matrix, then CE on top labels.  
L_Contr: supervised contrastive (Khosla 2020) using top-class as positive grouping, on L2-normalized z.

In [ ]:
class CrossEntropyLoss(nn.Module):
    def __init__(self, label_smoothing=0.01):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self, logits, labels, z=None, top_labels=None):
        l = self.ce(logits, labels)
        return l, {'ce': l.detach(),
                   'top': torch.tensor(0.0, device=logits.device),
                   'contr': torch.tensor(0.0, device=logits.device)}

class HierarchicalLoss(nn.Module):
    def __init__(self, sub2top_tensor, num_top_classes,
                 lambda_top=0.3, lambda_contr=0.1, tau=0.07, label_smoothing=0.01):
        super().__init__()
        sub2top = sub2top_tensor.long()
        n_sub = int(sub2top.numel()); n_top = int(num_top_classes)
        mask = torch.zeros(n_top, n_sub)
        for s in range(n_sub):
            mask[sub2top[s].item(), s] = 1.0
        self.register_buffer('top_mask', mask)
        self.register_buffer('sub2top', sub2top)
        self.lambda_top = float(lambda_top)
        self.lambda_contr = float(lambda_contr)
        self.tau = float(tau)
        self.ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    def forward(self, logits, labels, z=None, top_labels=None):
        l_ce = self.ce(logits, labels)
        if top_labels is not None:
            p_sub = F.log_softmax(logits, dim=1).exp()
            p_top = p_sub @ self.top_mask.t()
            l_top = F.nll_loss((p_top + 1e-12).log(), top_labels.long())
        else:
            l_top = torch.zeros((), device=logits.device)

        l_contr = torch.zeros((), device=logits.device)
        if (z is not None) and (top_labels is not None) and (z.size(0) > 1):
            zn = F.normalize(z, dim=1)
            sim = (zn @ zn.t()) / self.tau
            B = z.size(0)
            self_mask = torch.eye(B, dtype=torch.bool, device=z.device)
            sim_max = sim.masked_fill(self_mask, float('-inf')).max(dim=1, keepdim=True).values
            sim = sim - sim_max.detach()
            exp_sim = sim.exp().masked_fill(self_mask, 0.0)
            log_denom = exp_sim.sum(dim=1, keepdim=True).clamp(min=1e-12).log()
            log_prob = sim - log_denom
            pos = (top_labels.unsqueeze(0) == top_labels.unsqueeze(1)) & (~self_mask)
            cnt = pos.sum(dim=1).clamp(min=1).float()
            mean_lp_pos = (log_prob * pos.float()).sum(dim=1) / cnt
            valid = pos.any(dim=1)
            if valid.any():
                l_contr = -mean_lp_pos[valid].mean()

        total = l_ce + self.lambda_top * l_top + self.lambda_contr * l_contr
        return total, {'ce': l_ce.detach(), 'top': l_top.detach(), 'contr': l_contr.detach()}

## 6. Hierarchical evaluation metrics
`hierarchical_accuracy` (λ=0.5) + `hierarchical_prf_weighted` (λ=0.75) — same definitions as the official baseline `evaluate.py`.

In [ ]:
def hierarchical_accuracy(subcat, predictions_gt, lambda_param=0.5):
    s = []
    for prediction, gt in predictions_gt:
        if subcat == gt:
            ptop, psub = extend_subcat(prediction)
            gtop, gsub = extend_subcat(gt)
            if ptop == gtop and psub == gsub: s.append(1)
            elif ptop == gtop: s.append(lambda_param)
            else: s.append(0.0)
    return sum(s)/len(s) if s else float('nan')

def hierarchical_prf_weighted(subcat, predictions_gt, lambda_param=0.75):
    hPP, hRR = [], []
    for prediction, gt in predictions_gt:
        pi = extend_subcat(prediction); ti = extend_subcat(gt)
        inter = intersection(pi, ti)
        if subcat == prediction:
            w = 1 if prediction == gt else (lambda_param if get_top_level(prediction) == get_top_level(gt) else 0)
            hPP.append((w*len(inter))/len(pi))
        if subcat == gt:
            w = 1 if prediction == gt else (lambda_param if get_top_level(prediction) == get_top_level(gt) else 0)
            hRR.append((w*len(inter))/len(ti))
    cP = sum(hPP)/len(hPP) if hPP else 0
    cR = sum(hRR)/len(hRR) if hRR else 0
    cF = 0 if (cP == 0 and cR == 0) else 2*cP*cR/(cP+cR)
    return cP, cR, cF

def evaluate_test_loader(model, test_loader, device, class_dict, class_to_topclass):
    model.eval()
    preds, gts, scores, sids = [], [], [], []
    with torch.no_grad():
        for data in test_loader:
            y = data['class_idx'].to(device)
            audio = data.get('audio_embedding'); text = data.get('text_embedding')
            if audio is not None: audio = audio.to(device)
            if text is not None: text = text.to(device)
            _, logits, _ = model(audio, text)
            probs = torch.softmax(logits, dim=1)
            top1 = torch.argmax(probs, dim=1)
            ms = probs.gather(1, top1.unsqueeze(1)).squeeze(1)
            for i in range(y.size(0)):
                sid = data['sound_id'][i]
                if isinstance(sid, torch.Tensor): sid = sid.item()
                sids.append(sid); gts.append(y[i].item())
                preds.append(top1[i].item()); scores.append(float(ms[i]))
    id2c = build_id_to_class_mapping(class_dict)
    pl = [id2c.get(p, str(p)) for p in preds]
    gl = [id2c.get(g, str(g)) for g in gts]
    pairs = list(zip(pl, gl)); classes = list(set(gl))
    total = len(gts)
    correct = sum(p == g for p, g in zip(preds, gts))
    top_correct = sum(class_to_topclass.get(g) == class_to_topclass.get(p) for p, g in zip(preds, gts))
    macro_a, macro_t = [], []
    for c in classes:
        idx = [i for i, g in enumerate(gl) if g == c]
        if not idx: continue
        macro_a.append(sum(1 for i in idx if preds[i] == gts[i])/len(idx))
        macro_t.append(sum(1 for i in idx if class_to_topclass.get(gts[i]) == class_to_topclass.get(preds[i]))/len(idx))
    h_accs = []
    for c in classes:
        try:
            v = hierarchical_accuracy(c, pairs, 0.5)
            if not np.isnan(v): h_accs.append(v)
        except Exception: pass
    hPs, hRs, hFs = [], [], []
    for c in classes:
        try:
            p, r, f = hierarchical_prf_weighted(c, pairs, 0.75)
            if not (np.isnan(p) or np.isnan(r) or np.isnan(f)):
                hPs.append(p); hRs.append(r); hFs.append(f)
        except Exception: pass
    return {
        'accuracy': 100*correct/total if total else 0,
        'top_accuracy': 100*top_correct/total if total else 0,
        'macro_accuracy': 100*np.mean(macro_a) if macro_a else 0,
        'macro_top_accuracy': 100*np.mean(macro_t) if macro_t else 0,
        'hierarchical_accuracy': 100*np.mean(h_accs) if h_accs else 0,
        'hierarchical_precision': 100*np.mean(hPs) if hPs else 0,
        'hierarchical_recall': 100*np.mean(hRs) if hRs else 0,
        'hierarchical_f1': 100*np.mean(hFs) if hFs else 0,
    }

## 7. Training loop + experiment runner

In [ ]:
def train_one_run(train_loader, val_loader, criterion, *,
                  hidden_size, dropout, mode, num_classes, emb_audio, emb_text,
                  num_epochs=100, lr=1e-3, scheduler_type='step',
                  patience=5, early_stopping_factor=3, output_dir='./_run', verbose=True):
    os.makedirs(output_dir, exist_ok=True)
    model = BaseClassifier(hidden_size=hidden_size, num_classes=num_classes,
                           emb_size_audio=emb_audio, emb_size_text=emb_text,
                           dropout=dropout, use_batch_norm=True, mode=mode).to(DEVICE)
    init_weights(model)
    optim = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.StepLR(optim, step_size=20, gamma=0.5) if scheduler_type == 'step' else None

    best_val = 0.0; no_imp = 0
    history = defaultdict(list)

    for epoch in range(num_epochs):
        model.train()
        loss_sum = defaultdict(float); n_sum = 0
        for data in train_loader:
            y = data['class_idx'].to(DEVICE)
            top_y = data['top_class_idx'].to(DEVICE)
            audio = data.get('audio_embedding'); text = data.get('text_embedding')
            if audio is not None: audio = audio.to(DEVICE)
            if text is not None: text = text.to(DEVICE)
            optim.zero_grad()
            z, logits, _ = model(audio, text)
            loss, comp = criterion(logits, y, z=z, top_labels=top_y)
            loss.backward(); optim.step()
            B = y.size(0); n_sum += B
            loss_sum['total'] += loss.item() * B
            for k in ('ce', 'top', 'contr'):
                loss_sum[k] += float(comp[k]) * B

        model.eval(); correct = 0; total = 0
        with torch.no_grad():
            for data in val_loader:
                y = data['class_idx'].to(DEVICE)
                audio = data.get('audio_embedding'); text = data.get('text_embedding')
                if audio is not None: audio = audio.to(DEVICE)
                if text is not None: text = text.to(DEVICE)
                _, logits, _ = model(audio, text)
                pred = torch.argmax(logits, dim=1)
                correct += (pred == y).sum().item(); total += y.size(0)
        val_acc = 100 * correct / total
        for k in loss_sum: history[f'train_{k}_loss'].append(loss_sum[k]/n_sum)
        history['val_accuracy'].append(val_acc)
        history['lr'].append(optim.param_groups[0]['lr'])

        if verbose and (epoch < 3 or (epoch+1) % 10 == 0 or epoch == num_epochs-1):
            ls = ' | '.join(f"{k}:{loss_sum[k]/n_sum:.3f}" for k in ('total','ce','top','contr'))
            print(f"  ep{epoch+1:3d}/{num_epochs} val={val_acc:5.2f}% | {ls}")

        if sched is not None: sched.step()

        if val_acc > best_val:
            best_val = val_acc; no_imp = 0
            torch.save({
                'model_state': model.state_dict(),
                'config': {
                    'hidden_size': hidden_size, 'num_classes': num_classes,
                    'emb_size_audio': emb_audio, 'emb_size_text': emb_text,
                    'dropout': dropout, 'use_batch_norm': True, 'mode': mode,
                },
            }, os.path.join(output_dir, 'best_model.pth'))
        else:
            no_imp += 1
            if no_imp >= patience * early_stopping_factor:
                print(f"  early stop at epoch {epoch+1}"); break

    return best_val, history, model

def run_experiment(exp_id, exp_name, *,
                   conf_threshold=None, hier_loss=False,
                   lambda_top=0.3, lambda_contr=0.1, tau=0.07,
                   hidden_size=128, dropout=0.1,
                   epochs=100, batch_size=64, lr=1e-3,
                   k_folds=5, mode='both', seed=1821,
                   description=None, verbose=True):
    set_seed(seed)
    print('=' * 70); print(f"[{exp_id}] {exp_name}"); print('=' * 70)
    print(f"  conf>={conf_threshold} hier_loss={hier_loss} hidden={hidden_size} "
          f"dropout={dropout} lambda_top={lambda_top} lambda_contr={lambda_contr} tau={tau}")

    full_df = pd.read_csv(PROCESSED_CSV)
    if conf_threshold is not None:
        meta = pd.read_csv(METADATA_CSV)
        meta['sound_id'] = meta['sound_id'].astype(str).str.strip()
        keep = meta[meta['confidence'] >= conf_threshold]
        full_df['index'] = full_df['index'].astype(str)
        before = len(full_df)
        full_df = full_df[full_df['index'].isin(keep['sound_id'])].reset_index(drop=True)
        print(f"  conf-filter: {before} -> {len(full_df)} samples")

    labels = full_df['class_idx'].tolist()
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed)
    emb_audio = 512 if mode in ['audio', 'both'] else 0
    emb_text = 512 if mode in ['text', 'both'] else 0

    fold_results = []
    output_root = os.path.join(CODE_DIR, 'model_output', exp_name)
    os.makedirs(output_root, exist_ok=True)

    for fold, (trv_idx, te_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        print(f"\n-- Fold {fold} --")
        trv_lab = [labels[i] for i in trv_idx]
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        tr_rel, va_rel = next(sss.split(np.zeros(len(trv_lab)), trv_lab))
        tr_idx = [trv_idx[i] for i in tr_rel]; va_idx = [trv_idx[i] for i in va_rel]

        train_df = full_df.iloc[tr_idx].reset_index(drop=True)
        val_df = full_df.iloc[va_idx].reset_index(drop=True)
        test_df = full_df.iloc[te_idx].reset_index(drop=True)
        print(f"  train={len(train_df)} val={len(val_df)} test={len(test_df)}")

        train_loader = DataLoader(HATRDataset(train_df, aug=True, mask_pct=0.7),
                                  batch_size=batch_size, shuffle=True, drop_last=True,
                                  num_workers=0, pin_memory=torch.cuda.is_available())
        val_loader = DataLoader(HATRDataset(val_df, aug=False),
                                batch_size=batch_size, shuffle=False,
                                num_workers=0, pin_memory=torch.cuda.is_available())
        test_loader = DataLoader(HATRDataset(test_df, aug=False),
                                 batch_size=batch_size, shuffle=False,
                                 num_workers=0, pin_memory=torch.cuda.is_available())

        if hier_loss:
            criterion = HierarchicalLoss(SUB2TOP_TENSOR, NUM_TOP_CLASSES,
                                         lambda_top=lambda_top, lambda_contr=lambda_contr, tau=tau).to(DEVICE)
        else:
            criterion = CrossEntropyLoss().to(DEVICE)

        out = os.path.join(output_root, f'fold_{fold}')
        best_val, history, model = train_one_run(
            train_loader, val_loader, criterion,
            hidden_size=hidden_size, dropout=dropout, mode=mode,
            num_classes=NUM_CLASSES, emb_audio=emb_audio, emb_text=emb_text,
            num_epochs=epochs, lr=lr, output_dir=out, verbose=verbose,
        )
        ckpt = torch.load(os.path.join(out, 'best_model.pth'), map_location=DEVICE)
        eval_model = BaseClassifier(**ckpt['config']).to(DEVICE)
        eval_model.load_state_dict(ckpt['model_state'])
        metrics = evaluate_test_loader(eval_model, test_loader, DEVICE, CLASS_DICT, CLASS_TO_TOPCLASS)
        print(f"  fold {fold} -> H-Acc={metrics['hierarchical_accuracy']:.2f}% "
              f"acc={metrics['accuracy']:.2f}% top={metrics['top_accuracy']:.2f}%")
        fold_results.append({'fold': fold, **metrics, 'best_val': best_val})

    keys = ['accuracy','top_accuracy','macro_accuracy','macro_top_accuracy',
            'hierarchical_accuracy','hierarchical_precision','hierarchical_recall','hierarchical_f1']
    agg = {k: float(np.mean([r[k] for r in fold_results])) for k in keys}
    std = {k: float(np.std([r[k] for r in fold_results])) for k in keys}
    print(f"\n[{exp_id}] AGG: H-Acc={agg['hierarchical_accuracy']:.2f}% +- {std['hierarchical_accuracy']:.2f}%, "
          f"acc={agg['accuracy']:.2f}%, top={agg['top_accuracy']:.2f}%, macro2nd={agg['macro_accuracy']:.2f}%")

    BASELINE_HACC = 79.45
    delta = agg['hierarchical_accuracy'] - BASELINE_HACC
    payload = {
        'exp_id': exp_id,
        'description': description or exp_name,
        'config': {
            'conf_threshold': conf_threshold, 'hier_loss': hier_loss,
            'lambda_top': lambda_top, 'lambda_contr': lambda_contr, 'tau': tau,
            'hidden_size': hidden_size, 'dropout': dropout,
            'epochs': epochs, 'batch_size': batch_size, 'lr': lr,
            'k_folds': k_folds, 'mode': mode, 'seed': seed,
        },
        'results': {'fold_avg': agg, 'fold_std': std, 'fold_details': fold_results},
        'vs_baseline_h_acc': f"{delta:+.2f}%",
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M'),
    }
    out_json = os.path.join(EXPERIMENTS_DIR, f"{exp_id}_{exp_name}.json")
    with open(out_json, 'w') as f:
        json.dump(make_serializable(payload), f, indent=2, default=str)
    print(f"  saved -> {out_json}")
    return payload

## 8. EXP-001 — confidence ≥ 4 filter (no hier loss)
Paper A claim: +9% 2nd-level / +5% top. Expected H-Acc ~84-87%.

In [ ]:
exp001 = run_experiment(
    exp_id='exp_001', exp_name='conf4',
    description='confidence >= 4 filter, baseline architecture, plain CE',
    conf_threshold=4, hier_loss=False,
    hidden_size=128, dropout=0.1, epochs=100, batch_size=64, lr=1e-3,
    k_folds=5, mode='both', seed=1821,
)

## 9. EXP-005 — + L_Top + L_Contr hierarchical losses
Paper B initial values: λ_top=0.3, λ_contr=0.1, τ=0.07. Expected: +1-3% H-Acc on top of EXP-001.

In [ ]:
exp005 = run_experiment(
    exp_id='exp_005', exp_name='conf4_hier',
    description='conf>=4 + L_Top + L_Contr (lambda_top=0.3, lambda_contr=0.1, tau=0.07)',
    conf_threshold=4, hier_loss=True,
    lambda_top=0.3, lambda_contr=0.1, tau=0.07,
    hidden_size=128, dropout=0.1, epochs=100, batch_size=64, lr=1e-3,
    k_folds=5, mode='both', seed=1821,
)

## 10. EXP-006 — + hidden_size 256
More capacity. Expected +0.5-1% H-Acc if not overfit.

In [ ]:
exp006 = run_experiment(
    exp_id='exp_006', exp_name='conf4_hier_h256',
    description='conf>=4 + hier loss + hidden_size=256',
    conf_threshold=4, hier_loss=True,
    hidden_size=256, dropout=0.1, epochs=100, batch_size=64, lr=1e-3,
    k_folds=5, mode='both', seed=1821,
)

## 11. EXP-007 — + hidden_size 512
Largest capacity. May overfit on a 4k-train split — that's the experimental question.

In [ ]:
exp007 = run_experiment(
    exp_id='exp_007', exp_name='conf4_hier_h512',
    description='conf>=4 + hier loss + hidden_size=512',
    conf_threshold=4, hier_loss=True,
    hidden_size=512, dropout=0.15, epochs=100, batch_size=64, lr=1e-3,
    k_folds=5, mode='both', seed=1821,
)

## 12. Summary, CLAUDE.md update, git commit

In [ ]:
import subprocess

runs = [
    ('exp_000', 'baseline (reference)', 79.45, 88.88, 74.02),
    ('exp_001', 'conf>=4',
        exp001['results']['fold_avg']['hierarchical_accuracy'],
        exp001['results']['fold_avg']['top_accuracy'],
        exp001['results']['fold_avg']['macro_accuracy']),
    ('exp_005', 'conf>=4 + hier',
        exp005['results']['fold_avg']['hierarchical_accuracy'],
        exp005['results']['fold_avg']['top_accuracy'],
        exp005['results']['fold_avg']['macro_accuracy']),
    ('exp_006', 'conf>=4 + hier + h256',
        exp006['results']['fold_avg']['hierarchical_accuracy'],
        exp006['results']['fold_avg']['top_accuracy'],
        exp006['results']['fold_avg']['macro_accuracy']),
    ('exp_007', 'conf>=4 + hier + h512',
        exp007['results']['fold_avg']['hierarchical_accuracy'],
        exp007['results']['fold_avg']['top_accuracy'],
        exp007['results']['fold_avg']['macro_accuracy']),
]
print(f"{'ID':<10} {'Description':<28} {'H-Acc':>8} {'Top':>8} {'Macro2nd':>10} {'vs base':>10}")
print('-' * 80)
for eid, desc, h, t, m in runs:
    delta = h - 79.45
    print(f"{eid:<10} {desc:<28} {h:7.2f}% {t:7.2f}% {m:9.2f}% {delta:+9.2f}%")

best = max(runs[1:], key=lambda r: r[2])
print(f"\nBEST: {best[0]} ({best[1]}) -- H-Acc {best[2]:.2f}% ({best[2]-79.45:+.2f}% vs baseline)")

def append_to_claude_md():
    p = os.path.join(PROJECT_ROOT, 'CLAUDE.md')
    with open(p, 'r', encoding='utf-8') as f:
        content = f.read()
    new_rows = []
    for eid, desc, h, t, m in runs[1:]:
        delta = h - 79.45
        nid = eid.replace('exp_', '').lstrip('0') or '0'
        new_rows.append(f"| {nid:>3} | main | {desc} | **{h:.2f}%** | {m:.2f}% | {t:.2f}% | {delta:+.2f}% |")
    addition = '\n' + '\n'.join(new_rows) + '\n'
    marker = '| 000 | main | baseline (both mode, 5-fold) |'
    if marker in content and addition.strip() not in content:
        idx = content.index(marker)
        line_end = content.index('\n', idx) + 1
        content = content[:line_end] + addition + content[line_end:]
        with open(p, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f"appended {len(new_rows)} rows to CLAUDE.md")
    else:
        print('CLAUDE.md not modified')

append_to_claude_md()

def git(*args):
    r = subprocess.run(['git', *args], cwd=PROJECT_ROOT, capture_output=True, text=True)
    if r.stdout: print(r.stdout)
    if r.stderr: print(r.stderr)
    return r

git('add', 'experiments/', 'CLAUDE.md', 'DCASE2026_World1st.ipynb', 'dcase2026_task1_baseline/')
best_id, best_desc, best_h, _, _ = best
git('commit', '-m', f"[{best_id.upper().replace('_', '-')}] {best_desc} | H-Acc: {best_h:.2f}%")
git('push', 'origin', 'main')